In [7]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [8]:
import torch
import torch.nn as nn

from torchvision.models import resnet18, ResNet18_Weights

In [9]:
weights = ResNet18_Weights.DEFAULT
resnet = resnet18(weights=weights)

# replaces the final fully connected layer of the ResNet18 model with an identity layer, 
# effectively removing it and allowing the model to output feature embeddings instead of class predictions.
resnet.fc = nn.Identity() 
# Identity means instead of So instead of classifying the image, it will output a feature vector (learned features).

resnet

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_sta

In [10]:
class SiameseNetwork(nn.Module):

    def __init__(self):
        super().__init__()

        weights = ResNet18_Weights.DEFAULT
        self.backbone = resnet18(weights=weights)

        self.backbone.fc = nn.Identity()

    def forward_once(self, x):
        return self.backbone(x)

    def forward(self, img1, img2):

        embedding1 = self.forward_once(img1)
        embedding2 = self.forward_once(img2)

        return embedding1, embedding2

In [11]:
import sys
from pathlib import Path
from src.dataset import *


root_path = Path.cwd().parent
if str(root_path) not in sys.path:
    sys.path.append(str(root_path))

model = SiameseNetwork()
total = sum(p.numel() for p in model.parameters())

print(f"Total Parameters: {total:,}")

img1, img2, labels = next(iter(loader))

# Print to verify original shape
print(f"Original shape: {img1.shape}") 

# Correctly check if the channel dimension (index 1) is grayscale (1)
# if img1.shape[1] == 1:
#     img1 = img1.repeat(1, 3, 1, 1)
#     img2 = img2.repeat(1, 3, 1, 1)

print(f"Adjusted shape: {img1.shape}") # Should now show torch.Size([32, 3, 224, 224])


# Now the model will accept them without errors
print("\nPassing the images through the model...\n")
embedding1, embedding2 = model(img1, img2)

print(img1.shape)  # Should now show torch.Size([32, 3, 224, 224])
print(img2.shape)
print(embedding1.shape)
print(embedding2.shape)


RuntimeError: output with shape [1, 224, 224] doesn't match the broadcast shape [3, 224, 224]